# Multi-Agent Reinforcement Learning PettingZoo

In [8]:
from environment import TrafficEnvironment
from keychain import Keychain as kc
import os
from services import Trainer
from services import create_agent_objects
from services import confirm_env_variable
from services import get_json
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_checker import check_env
import ray
from ray import tune
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.env.wrappers.pettingzoo_env import ParallelPettingZooEnv
from ray.tune.registry import register_env
from pettingzoo.test import parallel_api_test
from gymnasium.spaces import Box, Discrete
from pettingzoo.classic import leduc_holdem_v4
from torch import nn
import gymnasium as gym
import supersuit as ss


confirm_env_variable(kc.SUMO_HOME, append="tools")
params = get_json(kc.PARAMS_PATH)

[CONFIRMED] Environment variable exists: SUMO_HOME
[SUCCESS] Added module directory: C:\Program Files (x86)\Eclipse\Sumo\tools


###  Initialization of the TrafficEnvironment 

- **Network**: initialize network, create demand, create paths, calculate free flow travel times.

- **Learning**: initialize observation and action space.

In [3]:
env = TrafficEnvironment(params[kc.SIMULATION_PARAMETERS])

[SUCCESS] Generated & saved 6 paths to: paths.csv
[SUCCESS] Environment initiated!


### Parallel API Test

To make sure our custom environment is consistent with the API, we have the api_test, more information [here](https://pettingzoo.farama.org/content/environment_tests/).

In [4]:
parallel_api_test(env, num_cycles=1_000_000)

joint_action is:  {'agent1': 2, 'agent2': 1} 

costs is:  [ 97. 153.] 


joint_action is:  {'agent1': 2, 'agent2': 1} 

costs is:  [ 97. 153.] 


Passed Parallel API test


### Stable-baselines3

The reset function returns dictionaries containing observations and info for each agent.

In [7]:
env.reset(seed=None)

({'agent1': array([0.53732474]), 'agent2': array([0.99161507])},
 {'agent1': {}, 'agent2': {}})